# Día 2 — Regularización, árboles de regresión y generalización

**Módulo 3: Machine Learning Aplicado — CEC EPN / Banco Central del Ecuador**  
**Fecha:** 14-10-2026

Este notebook está escrito como **material de estudio**, no como una colección de comandos. La regla será siempre:

> **pregunta → datos → exploración → preparación → modelo → métricas → diagnóstico → decisión**


## Objetivos

Hoy continuamos **el mismo problema económico** del Día 1. La secuencia es deliberada:

1. recordar por qué un modelo puede generalizar mal;
2. entender **Ridge, Lasso y Elastic Net** como extensiones de la regresión lineal;
3. pasar a **árboles de regresión**, que ya no obligan a una relación lineal;
4. entender **Random Forest** y **Gradient Boosting**;
5. comparar **XGBoost, LightGBM y CatBoost** a nivel práctico/conceptual;
6. usar Train/Validation para diagnosticar, no Test para “perseguir” resultados;
7. cerrar con el puente de regresión hacia clasificación.

> No se pretende memorizar todos los hiperparámetros en una noche. El objetivo es entender **qué problema intenta resolver cada familia y qué parámetro cambia su comportamiento**.

## 0. Librerías

La mayoría vienen con Anaconda/scikit-learn. Para los tres paquetes externos, si el import falla puedes instalar manualmente:

```python
%pip install xgboost lightgbm catboost
```

No hay instalación automática en el notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

# Modelos externos: se dejan como opcionales y NO se instalan automáticamente.
try:
    from xgboost import XGBRegressor
except ImportError:
    XGBRegressor = None

try:
    from lightgbm import LGBMRegressor
except ImportError:
    LGBMRegressor = None

try:
    from catboost import CatBoostRegressor
except ImportError:
    CatBoostRegressor = None

pd.set_option('display.max_columns', 40)

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')

# Recuperamos el dataset conductor y aplicamos la misma limpieza del Día 1.
df = pd.read_csv(DATA_DIR / 'actividad_economica_demo.csv')
df['fecha'] = pd.to_datetime(df['fecha'])
df = df.drop_duplicates().copy()

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())

print(df.shape)
df.head()

Seguimos con el mismo target: `crecimiento_actividad_pct`. Esto permite que las diferencias de métricas se deban principalmente al **modelo**, no a cambiar constantemente de problema.

# 1. Recordatorio: parámetros, hiperparámetros y generalización

En regresión lineal:

$$
\hat y=b_0+b_1x_1+\dots+b_px_p
$$

- $b_0,b_1,...$ son **parámetros** aprendidos por `fit()`;
- elegir variables, `alpha`, profundidad de un árbol o número de árboles son decisiones/ **hiperparámetros**.

Un modelo no es bueno porque memoriza Train. Queremos que funcione en datos que no vio.

In [ ]:
features = [
    'inversion_mill','precio_petroleo_usd','exportaciones_mill',
    'indice_confianza','inflacion_pct','desempleo_pct'
]
X = df[features]
y = df['crecimiento_actividad_pct']

# 70/15/15, igual que ayer.
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

print('Train', X_train.shape, 'Val', X_val.shape, 'Test', X_test.shape)

In [ ]:
def evaluar(nombre, modelo, Xtr=X_train, ytr=y_train, Xv=X_val, yv=y_val):
    """Entrena en Train y devuelve métricas comparables en Train y Validation."""
    modelo.fit(Xtr, ytr)
    pred_tr = modelo.predict(Xtr)
    pred_v = modelo.predict(Xv)
    return {
        'modelo': nombre,
        'MAE_train': mean_absolute_error(ytr, pred_tr),
        'MAE_val': mean_absolute_error(yv, pred_v),
        'RMSE_val': np.sqrt(mean_squared_error(yv, pred_v)),
        'R2_train': r2_score(ytr, pred_tr),
        'R2_val': r2_score(yv, pred_v)
    }

base_lineal = LinearRegression()
resultado_lineal = evaluar('LinearRegression', base_lineal)
pd.DataFrame([resultado_lineal]).round(3)

La diferencia entre Train y Validation ayuda a diagnosticar. No debemos mirar solo `R2_val`; también interesa cuánto se separa del rendimiento de Train y qué pasa con MAE/RMSE.

# 2. Ridge: reducir coeficientes grandes con penalización L2

Ridge modifica el objetivo:

$$
\text{Objetivo}=\text{SSE}+\alpha\sum_{j=1}^{p}b_j^2
$$

- `alpha = 0` se aproxima a regresión lineal sin penalización;
- `alpha` mayor presiona los coeficientes hacia 0;
- normalmente **no los vuelve exactamente 0**.

### ¿Por qué puede ayudar?

Si varias variables están correlacionadas, sus coeficientes pueden volverse inestables. Ridge acepta un poco más de error en Train a cambio de una solución más regular y potencialmente mejor generalización.

### Importante: escalar

La penalización depende del tamaño de los coeficientes. No queremos castigar más una variable solo porque está medida en miles en lugar de porcentajes. Por eso usamos `StandardScaler` dentro de un `Pipeline`.

In [ ]:
ridge = Pipeline([
    ('scale', StandardScaler()),
    ('model', Ridge(alpha=1.0))
])

pd.DataFrame([evaluar('Ridge alpha=1', ridge)]).round(3)

`Pipeline` garantiza que el `StandardScaler` aprende media/desviación **solo desde Train** cuando hacemos `fit()`. Así evitamos leakage.

## 2.1 Mover `alpha` y observar coeficientes

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
filas = []
coef_ridge = []

for alpha in alphas:
    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('model', Ridge(alpha=alpha))
    ])
    res = evaluar(f'Ridge {alpha}', pipe)
    filas.append({'alpha': alpha, 'MAE_val': res['MAE_val'], 'R2_val': res['R2_val']})
    coef_ridge.append(pipe.named_steps['model'].coef_)

pd.DataFrame(filas).round(4)

In [ ]:
# Camino de coeficientes: cómo se encogen al aumentar alpha.
coef_ridge = np.array(coef_ridge)
plt.figure(figsize=(9, 5))
for j, nombre in enumerate(features):
    plt.plot(alphas, coef_ridge[:, j], marker='o', label=nombre)
plt.xscale('log')
plt.xlabel('alpha (escala log)')
plt.ylabel('Coeficiente sobre variables estandarizadas')
plt.title('Ridge: alpha mayor -> coeficientes más pequeños')
plt.legend(fontsize=8)
plt.show()

### Qué observar

El gráfico no dice “qué variable causa el crecimiento”. Muestra cómo la **solución predictiva lineal** redistribuye/encoge coeficientes cuando aumentamos la penalización.

# 3. Lasso: penalización L1 y coeficientes exactamente 0

Lasso usa:

$$
\text{Objetivo}=\text{SSE}+\alpha\sum_{j=1}^{p}|b_j|
$$

La geometría de la penalización L1 hace posible que algunos coeficientes lleguen exactamente a 0. Por eso Lasso puede funcionar también como **selección automática de variables**.

Esto no significa que una variable con coeficiente 0 sea “inútil para siempre”; puede estar redundada por otras variables o su aporte lineal adicional puede ser pequeño en este dataset.

In [ ]:
lasso = Pipeline([
    ('scale', StandardScaler()),
    ('model', Lasso(alpha=0.05, max_iter=20000))
])
lasso.fit(X_train, y_train)

coef_lasso = pd.Series(
    lasso.named_steps['model'].coef_,
    index=features,
    name='coef_lasso'
)
coef_lasso

Las variables con coeficiente exactamente `0.0` fueron eliminadas por esa combinación de datos y `alpha`. Si cambiamos `alpha`, la selección puede cambiar.

In [ ]:
filas=[]
for alpha in [0.001,0.005,0.01,0.03,0.05,0.1,0.3,1.0]:
    pipe=Pipeline([
        ('scale',StandardScaler()),
        ('model',Lasso(alpha=alpha,max_iter=20000))
    ])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_val)
    coefs=pipe.named_steps['model'].coef_
    filas.append({
        'alpha':alpha,
        'variables_no_cero':int(np.sum(np.abs(coefs)>1e-10)),
        'MAE_val':mean_absolute_error(y_val,pred),
        'R2_val':r2_score(y_val,pred)
    })
pd.DataFrame(filas).round(4)

Fíjate en el compromiso: al aumentar `alpha`, el modelo se simplifica. Si simplificamos demasiado, Validation empeora: eso sería **underfitting por exceso de regularización**.

# 4. Elastic Net: combinar L1 y L2

Elastic Net combina las dos penalizaciones:

$$
\text{Objetivo}=\text{SSE}+\alpha\left[
\text{l1\_ratio}\sum|b_j|+(1-\text{l1\_ratio})\sum b_j^2
\right]
$$

- `l1_ratio = 1` se acerca a Lasso;
- `l1_ratio = 0` se acerca a Ridge;
- valores intermedios mezclan selección y estabilidad.

In [ ]:
elastic = Pipeline([
    ('scale', StandardScaler()),
    ('model', ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=20000))
])

pd.DataFrame([evaluar('ElasticNet', elastic)]).round(3)

In [ ]:
# Tabla comparativa lineal y regularizados con configuraciones didácticas.
modelos_lineales = {
    'Linear': LinearRegression(),
    'Ridge': Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=10))]),
    'Lasso': Pipeline([('scale',StandardScaler()),('model',Lasso(alpha=.03,max_iter=20000))]),
    'ElasticNet': Pipeline([('scale',StandardScaler()),('model',ElasticNet(alpha=.03,l1_ratio=.5,max_iter=20000))])
}

resultados = [evaluar(nombre, m) for nombre,m in modelos_lineales.items()]
pd.DataFrame(resultados).round(3).sort_values('MAE_val')

### Qué conclusión sí podemos sacar

Podemos decir qué configuración **generaliza mejor en esta Validation** según las métricas elegidas. No podemos decir que ese modelo será universalmente mejor ni que sus coeficientes son efectos causales.

# 5. Árbol de regresión: dejar de exigir una recta

Un árbol divide el espacio con reglas como:

```text
¿inversion_mill <= 105?
    sí  -> otra pregunta
    no  -> otra pregunta
```

Para regresión, una hoja suele predecir el **promedio de y de las observaciones de Train que llegaron a esa hoja**.

El árbol busca cortes que reduzcan el error de los grupos resultantes. Conceptualmente, con MSE:

$$
Error_{corte}=\frac{n_L}{n}MSE_L+\frac{n_R}{n}MSE_R
$$

El algoritmo CART es greedy: escoge la mejor división local en cada paso.

In [ ]:
# Árbol pequeño para poder visualizarlo.
arbol_pequeno = DecisionTreeRegressor(max_depth=3, min_samples_leaf=8, random_state=42)
arbol_pequeno.fit(X_train, y_train)

plt.figure(figsize=(18, 8))
plot_tree(
    arbol_pequeno,
    feature_names=features,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Árbol de regresión (profundidad 3)')
plt.show()

### Cómo leer un nodo

En cada caja aparecen, entre otros valores:

- la condición de corte;
- `squared_error`: variabilidad de y dentro del nodo;
- `samples`: cuántas filas llegaron;
- `value`: predicción promedio del nodo/hoja.

A diferencia de la regresión lineal, el árbol puede capturar interacciones y relaciones por tramos sin especificarlas manualmente.

In [ ]:
# La profundidad controla capacidad y riesgo de sobreajuste.
filas=[]
for depth in [1,2,3,4,6,10,None]:
    m=DecisionTreeRegressor(max_depth=depth,min_samples_leaf=2,random_state=42)
    m.fit(X_train,y_train)
    filas.append({
        'max_depth':str(depth),
        'R2_train':r2_score(y_train,m.predict(X_train)),
        'R2_val':r2_score(y_val,m.predict(X_val)),
        'MAE_val':mean_absolute_error(y_val,m.predict(X_val))
    })
pd.DataFrame(filas).round(3)

Busca el patrón típico: a mayor profundidad, Train casi siempre mejora. Validation puede mejorar hasta cierto punto y luego empeorar. Eso es una demostración directa de **overfitting por complejidad**.

# 6. Random Forest Regressor: muchos árboles y promedio

Un Random Forest no es “un árbol grande”. Construye muchos árboles diferentes:

1. cada árbol suele entrenarse con una muestra **bootstrap** de filas;
2. en cada división puede considerar solo un subconjunto de variables;
3. cada árbol produce una predicción;
4. el bosque **promedia** las predicciones.

La diversidad reduce la varianza respecto a un árbol individual.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=300,       # número de árboles
    max_depth=None,         # cada árbol puede crecer, limitado por otras reglas
    min_samples_leaf=4,     # evita hojas con muy pocos casos
    max_features='sqrt',    # aumenta diversidad entre árboles
    random_state=42,
    n_jobs=-1
)

pd.DataFrame([evaluar('RandomForest', forest)]).round(3)

### Hiperparámetros clave

- `n_estimators`: más árboles suele estabilizar, con mayor costo;
- `max_depth`: complejidad por árbol;
- `min_samples_leaf`: evidencia mínima en cada hoja;
- `max_features`: cuántas variables compiten en cada corte;
- `bootstrap`: muestreo con reemplazo.

Más árboles no sobreajustan exactamente igual que más profundidad, pero tampoco “arreglan” un mal diseño de variables.

In [ ]:
# Importancia interna del bosque.
forest.fit(X_train,y_train)
imp = pd.Series(forest.feature_importances_, index=features).sort_values(ascending=False)
imp

Esta `feature_importances_` mide cuánto se usaron variables para reducir el criterio dentro de los árboles. **No es causalidad** y puede sesgarse hacia variables con más oportunidades de corte. En proyectos serios conviene contrastarla con permutation importance o SHAP.

# 7. Gradient Boosting Regressor: árboles que corrigen errores

Random Forest construye árboles principalmente en paralelo y promedia. **Gradient Boosting** trabaja secuencialmente.

Idea simplificada:

1. empezamos con una predicción básica, por ejemplo la media;
2. calculamos residuos;
3. un árbol intenta predecir esos errores;
4. agregamos una fracción de su corrección;
5. repetimos.

$$
\hat y_{nuevo}=\hat y_{actual}+\text{learning\_rate}\times correccion
$$

`learning_rate` pequeño implica pasos pequeños y suele requerir más árboles.

In [ ]:
gbr = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=2,
    min_samples_leaf=5,
    random_state=42
)

pd.DataFrame([evaluar('GradientBoosting', gbr)]).round(3)

In [ ]:
# Relación learning_rate / n_estimators.
filas=[]
for lr, n in [(0.2,50),(0.1,100),(0.05,200),(0.03,300),(0.01,800)]:
    m=GradientBoostingRegressor(n_estimators=n,learning_rate=lr,max_depth=2,random_state=42)
    r=evaluar(f'lr={lr},n={n}',m)
    filas.append({'learning_rate':lr,'n_estimators':n,'R2_train':r['R2_train'],'R2_val':r['R2_val'],'MAE_val':r['MAE_val']})
pd.DataFrame(filas).round(3)

Un `learning_rate` bajo **no garantiza** ausencia de overfitting. Con suficientes árboles el modelo también puede empezar a corregir ruido de Train. Por eso Validation sigue siendo necesaria.

# 8. XGBoost: boosting con regularización y optimizaciones

XGBoost conserva la idea de árboles secuenciales, pero añade regularización, controles de división y una implementación altamente optimizada.

Hiperparámetros que conviene reconocer:

- `n_estimators`: número de árboles;
- `learning_rate`: tamaño del paso;
- `max_depth`: complejidad;
- `min_child_weight`: evidencia mínima en un hijo;
- `gamma`: mejora mínima para aceptar una división;
- `subsample`: fracción de filas;
- `colsample_bytree`: fracción de variables;
- `reg_alpha` / `reg_lambda`: regularización L1/L2.

In [ ]:
if XGBRegressor is None:
    print('XGBoost no está instalado. Si deseas ejecutar este bloque: %pip install xgboost')
else:
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )
    display(pd.DataFrame([evaluar('XGBoost', xgb)]).round(3))

No necesitamos “tunear” veinte parámetros hoy. Primero hay que entender qué controla cada familia: complejidad, muestreo, regularización o velocidad de aprendizaje.

# 9. LightGBM: crecimiento eficiente de árboles

LightGBM también es gradient boosting. Una diferencia importante es que suele crecer **leaf-wise**: expande primero la hoja que ofrece mayor ganancia, lo que puede ser muy eficiente pero también exige controlar complejidad.

Parámetros importantes:

- `num_leaves`: número máximo de hojas;
- `max_depth`: límite adicional;
- `min_child_samples`: observaciones mínimas por hoja;
- `learning_rate`, `n_estimators`;
- `subsample`, `colsample_bytree`;
- `reg_alpha`, `reg_lambda`.

In [ ]:
if LGBMRegressor is None:
    print('LightGBM no está instalado. Si deseas ejecutar este bloque: %pip install lightgbm')
else:
    lgbm = LGBMRegressor(
        n_estimators=300,
        learning_rate=0.03,
        num_leaves=15,
        max_depth=5,
        min_child_samples=10,
        random_state=42,
        verbosity=-1
    )
    display(pd.DataFrame([evaluar('LightGBM', lgbm)]).round(3))

En datasets pequeños, permitir demasiadas hojas puede crear reglas demasiado específicas. `num_leaves` y `min_child_samples` son controles muy relevantes.

# 10. CatBoost: especial atención a variables categóricas

CatBoost pertenece a la familia de boosting, pero fue diseñado con tratamiento sofisticado de variables categóricas.

En una regresión lineal hicimos one-hot manualmente. CatBoost puede recibir columnas como `region` y `sector` como categorías nativas.

Eso es especialmente útil cuando existen muchas categorías, aunque no elimina la necesidad de validar calidad de datos y leakage.

In [ ]:
if CatBoostRegressor is None:
    print('CatBoost no está instalado. Si deseas ejecutar este bloque: %pip install catboost')
else:
    features_cat = features + ['region','sector']
    Xc = df[features_cat].copy()
    yc = y.copy()

    # CatBoost necesita los índices de las columnas categóricas.
    cat_idx = [Xc.columns.get_loc('region'), Xc.columns.get_loc('sector')]

    Xc_train, Xc_temp, yc_train, yc_temp = train_test_split(Xc,yc,test_size=.30,random_state=42)
    Xc_val, Xc_test, yc_val, yc_test = train_test_split(Xc_temp,yc_temp,test_size=.50,random_state=42)

    cat = CatBoostRegressor(
        iterations=300,
        learning_rate=0.03,
        depth=5,
        loss_function='RMSE',
        random_seed=42,
        verbose=False
    )
    cat.fit(Xc_train,yc_train,cat_features=cat_idx)
    pred=cat.predict(Xc_val)
    print('MAE Validation:', round(mean_absolute_error(yc_val,pred),3))
    print('RMSE Validation:', round(np.sqrt(mean_squared_error(yc_val,pred)),3))
    print('R2 Validation:', round(r2_score(yc_val,pred),3))

### Diferencia conceptual útil

- **Linear/Ridge/Lasso/Elastic Net:** combinación lineal explícita y altamente interpretable.
- **Decision Tree:** reglas por tramos.
- **Random Forest:** promedio de muchos árboles diversos.
- **Gradient Boosting:** árboles secuenciales que corrigen errores.
- **XGBoost/LightGBM/CatBoost:** variantes especializadas de boosting con distintas estrategias de regularización, crecimiento y tratamiento de variables.

# 11. Comparar modelos con una tabla común

Una comparación justa usa:

- el mismo objetivo;
- el mismo Train/Validation;
- las mismas métricas;
- preprocesamiento apropiado para cada modelo.

In [ ]:
modelos = {
    'Linear': LinearRegression(),
    'Ridge': Pipeline([('scale',StandardScaler()),('model',Ridge(alpha=10))]),
    'Lasso': Pipeline([('scale',StandardScaler()),('model',Lasso(alpha=.03,max_iter=20000))]),
    'ElasticNet': Pipeline([('scale',StandardScaler()),('model',ElasticNet(alpha=.03,l1_ratio=.5,max_iter=20000))]),
    'DecisionTree': DecisionTreeRegressor(max_depth=5,min_samples_leaf=5,random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=250,min_samples_leaf=4,random_state=42,n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=300,learning_rate=.03,max_depth=2,random_state=42)
}

tabla=[]
for nombre,m in modelos.items():
    tabla.append(evaluar(nombre,m))

tabla_modelos = pd.DataFrame(tabla).sort_values('MAE_val')
tabla_modelos.round(3)

### Cómo decidir

No es correcto escoger automáticamente la primera fila. Pregunta también:

- ¿la mejora es grande o marginal?
- ¿el modelo es estable entre splits/CV?
- ¿necesito explicar cada predicción?
- ¿cuánto cuesta entrenarlo/mantenerlo?
- ¿hay riesgo de leakage?
- ¿cómo se comporta por región/sector?

El “mejor” modelo depende del objetivo operativo, no solo de una métrica.

# 12. Puente hacia clasificación

Hasta ahora y era un número continuo: **regresión**.

Si transformamos el problema en una categoría, por ejemplo:

```text
crecimiento_actividad_pct > 2  -> crecimiento_alto = 1
resto                         -> crecimiento_alto = 0
```

pasamos a **clasificación**. La familia de árboles sigue existiendo, pero ahora las hojas predicen clases/probabilidades y los criterios de división miden impureza (Gini/Entropy) en lugar de error de regresión.

En el Día 3 trabajaremos clasificación con un dataset bancario y veremos matriz de confusión, Accuracy, Precision, Recall, F1, KNN y SVM.

# Preguntas de comprensión

1. ¿Por qué Ridge suele requerir escalado?
2. ¿Qué diferencia conceptual existe entre Ridge y Lasso?
3. ¿Qué controla `l1_ratio` en Elastic Net?
4. ¿Cómo obtiene un árbol de regresión el valor de una hoja?
5. ¿Por qué un árbol profundo puede tener Train excelente y Validation mala?
6. ¿Qué hace diferente a Random Forest frente a un solo árbol?
7. ¿Qué significa que Gradient Boosting “corrija residuos”?
8. ¿Por qué `learning_rate` bajo no elimina automáticamente overfitting?
9. ¿Qué ventajas conceptuales ofrecen XGBoost, LightGBM y CatBoost?
10. ¿Por qué no podemos llamar causal a una feature importance alta?